# 第7回：ニューラルネットワーク基礎 〜 脳を模した数理モデルの探求 〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session07_neural_networks.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

---

## 📋 本日の学習ロードマップ (計90分)

1. **【講義1】人工ニューロンの誕生 (10分)**
   - 生物学的ニューロンからパーセプトロンへ。重み $w$ とバイアス $b$。
2. **【講義2】活性化関数：なぜ「非線形」が必要か (15分)**
   - Sigmoid, ReLU, Softmax の使い分け。勾配消失問題の入り口。
3. **【実習1】活性化関数を可視化して特性を知る (10分)**
   - 数式とグラフの対応を確認する。
4. **【講義3】多層構造：ディープラーニングへの進化 (15分)**
   - 入力層・隠れ層・出力層。全結合層の仕組み。
5. **【講義4】学習の心臓部：バックプロパゲーション (15分)**
   - 誤差逆伝播。連鎖律。Adam オプティマイザの凄さ。
6. **【実習2】Keras/TensorFlow による MLP 構築 (15分)**
   - 手書き文字認識 (MNIST) を用いた「深層」体験。
7. **【総合演習】「最適な層」を設計する実験 (10分)**

---

## 1. セットアップ

現代の AI 開発のデファクトスタンダード、TensorFlow とそのラッパー Keras を使用します。

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='white', context='talk')

print(f"TensorFlow Version: {tf.__version__}")

## 2. 【講義1+2】人工ニューロンと活性化関数

### 2.1 ユニット内部で何が起きているか？
1 つのニューロンは、複数の入力 $x$ に対し「重要度（重み $w$）」をかけ、合計値を出します。そこに「発火のしやすさ（バイアス $b$）」を足したものが、次の層への信号になります。

### 2.2 非線形性の必要性
どんなに層を重ねても、単純な足し算・かけ算だけでは「ただの一次関数（直線）」しか表現できません。複雑な境界線を描くためには、出力をグニャリと曲げる **活性化関数** が不可欠です。
- **ReLU (Rectified Linear Unit)**: 0 以下は 0、0 以上はそのまま。現代の主力。
- **Sigmoid**: 0 〜 1 に押し込める。二値分類の最後（出力層）に使われる。
- **Softmax**: 合計を 1 にする。多クラス分類の最後で「確率」を出すために使われる。

---

### 【実習1】活性化関数を描いてみよう

In [ ]:
z = np.linspace(-5, 5, 200)
def sigmoid(x): return 1 / (1 + np.exp(-x))
def relu(x): return np.maximum(0, x)
def tanh(x): return np.tanh(x)

plt.figure(figsize=(10, 5))
plt.plot(z, sigmoid(z), label='Sigmoid')
plt.plot(z, relu(z), label='ReLU', linestyle='--')
plt.plot(z, tanh(z), label='Tanh')
plt.axvline(0, color='gray', lw=1)
plt.axhline(0, color='gray', lw=1)
plt.title("Comparison of Activation Functions")
plt.legend()
plt.show()

## 3. 【講義3+4】バックプロパゲーションの論理

### 3.1 誤差逆伝播 (Backpropagation)
1 つの MLP には何万、何億という「重み」があります。これを人間が調整するのは不可能です。AI は「予測と正解の誤差」を出発点として、**合成関数の微分（連鎖律）**を使い、出力側から入力側へと「重みをどれだけ修正すべきか」を伝播させていきます。

### 3.2 Adam オプティマイザ
勾配（傾き）を下っていく際、「今までの勢い（慣性）」と「学習率の自動調整」を組み合わせたアルゴリズムです。現代の最適化手法で最も信頼されています。

---

## 4. 【実習2】MNIST：手書き文字 6 万枚の学習

画像データ (28x28 = 784次元) を、1 列のベクトルとして扱い、全結合層 (Dense Layer) に通します。

In [ ]:
# 1. データのロードと正規化 (0〜255 -> 0〜1)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255
x_test = x_test.reshape(-1, 784).astype("float32") / 255

# 2. モデル構造の定義
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(784,)),
    layers.Dropout(0.2), # 過学習を防ぐための工夫！
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax') # 0〜9 の 10クラス確率を出力
])

# 3. コンパイル（エンジン設定）
model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', # 多クラス用損失
              metrics=['accuracy'])

# 4. 学習の実行
history = model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.2, verbose=1)

## 5. 学習結果の多角的評価

In [ ]:
# 学習曲線の可視化
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title("Accuracy Curve")
plt.legend()
plt.show()

# 個別のテスト予測
sample_idx = 0
test_sample = x_test[sample_idx].reshape(1, 784)
prediction = model.predict(test_sample)
print(f"\nAIの予測: {np.argmax(prediction[0])}")
print(f"正解: {y_test[sample_idx]}")

---

## ✏️ 本日の最終研究ミッション (Neural Architect)

**シナリオ**: あなたは、手書き文字認識の精度を極限まで高める責任者に選ばれました。

### 課題 1: 特徴空間の崩壊
もし `layers.Dense` のユニット数を 10 などの極端に小さい数字に設定して 2 層重ねた場合、精度はどう変化しますか？ 計算コストと精度の「トレードオフ」を観察してください。

### 課題 2: 活性化関数の実験
全ての `activation='relu'` を `activation='sigmoid'` に置き換えて再学習させてください。学習速度や最終精度にどのような違いが現れますか？ これが「ReLU が支持される理由」へのヒントになります。

### 課題 3: ミス分析 (Fail Analysis)
テストデータの中で、AI が間違えて予測してしまった画像を 10 枚ほど表示させてください。人間の目で見ると、AI がなぜ間違えたのか（形が崩れているのか、他の数字に見えるのか）を推論してください。

---

In [ ]:
# ここに回答を記述してください



---
**Last updated**: 2026-02-15
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)